Step 1: Grab raw drawings from MongoDB database

In [1]:
import os
from pathlib import Path

from PIL import Image
import pymongo as pm
from dotenv import load_dotenv
load_dotenv()
from vislearnlabpy.extractions.drawingtask_extractor import DrawingTaskExtractor
from vislearnlabpy.drawings.svg_render_helpers import StrokeSettings

/Scratch/nsahrling/miniconda3/envs/vll-312/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Fill out the fields below. Also make sure to fill in your username and password into an auth.txt file, using a similar format to how the auth_template.txt file is set up, before running the commands below.

In [2]:
# make sure to add password to a .env file
DATABASE_NAME = "kiddraw"
COLLECTION_NAME = "birch_run_v1" # collections to play around with: "birch_run_v1", "cdm_hybrid_v1", "cdm_run_v8", "india_run_v1"
SAVE_DIR = "birch_draw" # often set to /Volumes/vislearnlab/experiments/drawings/data..

This is the Mongo extractor

In [3]:
# do not commit this auth.txt file!
auth_file = "auth.txt"
with open(auth_file, "r") as f:
    mongo_connection_string = f.readline().strip()
# use a different extractor object for different collections
birch_extractor = DrawingTaskExtractor(mongo_connection_string, DATABASE_NAME, "birch_run_v1", SAVE_DIR)

Can be used to directly prompt Mongo with PyMongo

In [6]:
from pymongo import MongoClient, errors

test_client = MongoClient(
    mongo_connection_string,
    serverSelectionTimeoutMS=5000,   
    connectTimeoutMS=3000,
    socketTimeoutMS=3000,
)
try:
    print("Ping:", test_client.admin.command("ping"))
except errors.ServerSelectionTimeoutError as e:
    print("Server selection failed \n", e)

Server selection failed 
 dbaas-db-3323175-6ea38d7c.mongo.ondigitalocean.com:27017: timed out (configured timeouts: socketTimeoutMS: 3000.0ms, connectTimeoutMS: 3000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 69a88ab3462c4d97f595717d, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('dbaas-db-3323175-6ea38d7c.mongo.ondigitalocean.com', 27017) server_type: Unknown, rtt: None, error=NetworkTimeout('dbaas-db-3323175-6ea38d7c.mongo.ondigitalocean.com:27017: timed out (configured timeouts: socketTimeoutMS: 3000.0ms, connectTimeoutMS: 3000.0ms)')>]>


In [ ]:
all_images = list(birch_extractor.collection.find({'$and': [{'dataType':'finalImage'}]}))
print(f"Total final images in {birch_extractor.collection.name}: {len(all_images)}")

KeyboardInterrupt: 

Or can extract full datasets as needed. Audio can also be extracted but we have transcribed it by hand for ease of access.

In [14]:
# transform_file 
birch_extractor.extract_images(transform_file=True)

Drawing sessions processed:  12%|█▏        | 12/101 [05:00<39:42, 26.77s/it]

Wrote 100 images in 5.3 minutes


Drawing sessions processed:  23%|██▎       | 23/101 [07:36<12:23,  9.53s/it]

Blank image - skipping crop


Drawing sessions processed:  26%|██▌       | 26/101 [08:50<23:53, 19.11s/it]

Blank image - skipping crop
Blank image - skipping crop


... (10 outputs left out)

Drawing sessions processed:  92%|█████████▏| 93/101 [38:02<02:16, 17.01s/it]

Wrote 600 images in 38.4 minutes


Drawing sessions processed: 100%|██████████| 101/101 [41:29<00:00, 24.65s/it]

Finished processing 664 image files


In [ ]:
# Extract all stroke PNGs across all participants and categories
birch_extractor.extract_strokes(save_type="png")

Extract stroke-by-stroke PNGs for all participants and categories. These are used in the time analysis pipeline.